In [1]:
#import required libraries
import kagglehub
import pandas as pd
import numpy as np
import os
import torch
from collections import defaultdict
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
from tqdm import tqdm

In [2]:
#download the dataset
path = kagglehub.dataset_download("prajitdatta/movielens-100k-dataset")
possible_path = os.path.join(path, "ml-100k", "u.data")
if not os.path.exists(possible_path):
    possible_path = os.path.join(path, "u.data")

#select relevant columns
rating_columns = ["user_id", "item_id", "rating", "timestamp"]
df = pd.read_csv(possible_path, sep="\t", names=rating_columns)

#get number of items in dataset
df["user_id"] = df["user_id"] - 1
num_items = df["item_id"].max() + 1

100%|██████████| 4.77M/4.77M [00:01<00:00, 3.72MB/s]

Extracting files...


In [3]:
#sort by user and timestamp
df = df.sort_values(by=["user_id", "timestamp"])

#every user gets a list of his/her movies
user_group = df.groupby("user_id")["item_id"].apply(list)

#create train and test data
train_sequences = []
test_data = []

for user_id, items in user_group.items():
    target = items[-1]
    history = items[:-1]

    test_data.append((history, target))
    train_sequences.append(history)

In [4]:
#here we do some data preprocessing

#determines how many of the last n movies we will consider
max_len = 20

#to add padding
def pad_seq(seq, max_len):
    return [0] * (max_len - len(seq)) + seq

rnn_x = []
rnn_y = []

#we need to create more training data for the RNN,so we use sliding windows
for seq in train_sequences:
    for i in range(1, len(seq)):
        inp = seq[:i][-max_len:]
        target = seq[i]

        rnn_x.append(pad_seq(inp, max_len))
        rnn_y.append(target)

#transform to tensors
train_tensor_x = torch.tensor(rnn_x, dtype=torch.long)
train_tensor_y = torch.tensor(rnn_y, dtype=torch.long)

#our dataloader
rnn_loader = DataLoader(TensorDataset(train_tensor_x, train_tensor_y), batch_size=128, shuffle=True)

In [5]:
#this is the simple GRU model
class GRURecommender(nn.Module):
    def __init__(self, num_items, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.item_embedding = nn.Embedding(num_items, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_items)

    def forward(self, seq):
        x = self.item_embedding(seq)
        out, h = self.gru(x)
        return self.fc(out[:, -1, :])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rnn_model = GRURecommender(num_items).to(device)
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

#train the RNN
rnn_model.train()
epochs = 10
for epoch in range(epochs):
    total_loss = 0
    for x_batch, y_batch in rnn_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = rnn_model(x_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss {total_loss/len(rnn_loader):.4f}")

static_counts = defaultdict(lambda: defaultdict(int))
markov_counts = defaultdict(lambda: defaultdict(int))

#for the markov chain, we want to use at max 5 movies as the context
max_markov = 5

#train CF and Markov model
for seq in train_sequences:
    unique_items = set(seq)
    for i in unique_items:
        for j in unique_items:
            if i != j: static_counts[i][j] += 1
    for i in range(len(seq) - 1):
        target = seq[i+1]
        for k in range(1, max_markov + 1):
            if i - k + 1 >= 0:
                context = tuple(seq[i-k+1 : i+1])
                markov_counts[context][target] += 1


Epoch 1: Loss 6.6423
Epoch 2: Loss 6.0149
Epoch 3: Loss 5.8429
Epoch 4: Loss 5.7517
Epoch 5: Loss 5.6782
Epoch 6: Loss 5.6146
Epoch 7: Loss 5.5586
Epoch 8: Loss 5.5078
Epoch 9: Loss 5.4599
Epoch 10: Loss 5.4136


In [6]:

#this function gives us the top k movies
def get_top_k(scores, history, top_k):
    candidates = sorted(scores, key=scores.get, reverse=True)
    candidates = [x for x in candidates if x not in history]
    return candidates[:top_k]

#get CF predictions
def predict_static(history, top_k=10):
    scores = defaultdict(int)
    for item in history:
        if item in static_counts:
            for neighbor, count in static_counts[item].items():
                scores[neighbor] += count
    return get_top_k(scores, history, top_k)

#get Markov model predictions
def predict_markov_variable(history, top_k=10):
    preds = []
    for order in range(max_markov, 0, -1):
        if len(history) >= order:
            context = tuple(history[-order:])
            if context in markov_counts:
                new_candidates = get_top_k(markov_counts[context], history, top_k)
                for item in new_candidates:
                    if item not in preds:
                        preds.append(item)
                        if len(preds) >= top_k: return preds

    return preds

#get RNN predictions
def predict_rnn(history, top_k=10):
    rnn_model.eval()
    seq_in = pad_seq(history[-max_len:], max_len)
    seq_tensor = torch.tensor([seq_in], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = rnn_model(seq_tensor)
        _, top_indices = torch.topk(logits, top_k)
        return top_indices.cpu().numpy()[0]

#
hits = {"CF": 0, "Markov": 0, "RNN": 0}
total = len(test_data)

for history, target in tqdm(test_data):
    if target in predict_static(history): hits["CF"] += 1
    if target in predict_markov_variable(history): hits["Markov"] += 1
    if target in predict_rnn(history): hits["RNN"] += 1


print(f"\n Collaborative Filtering:{hits["CF"] / total:.4f}")
print(f"\n Markov Models:{hits["Markov"] / total:.4f}")
print(f"RNN:{hits["RNN"] / total:.4f}")


100%|██████████| 943/943 [00:21<00:00, 43.42it/s]


 Collaborative Filtering:0.0912

 Markov Models:0.1357
RNN:0.1421
